# Paper shadow — living GEX/VEX/squeeze monitor

**Harness:** [`../../applications/paper_shadow/`](../../applications/paper_shadow/) · `live_orders=false`  
Telemetry only until Promote (still **0**). Wires Hold monitors; never flips TOB-cross to α.


In [ ]:
from pathlib import Path
import json, os, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

BOOK = Path(os.environ.get("ARES_MICROSTRUCTURE") or (Path.home() / "srv" / "ares-microstructure")) / "research" / "books" / "squeeze_metrics"
OUT = BOOK / "out"
SCRIPTS = BOOK / "scripts"
ROOT = BOOK.parents[2]
sys.path.insert(0, str(SCRIPTS))
sys.path.insert(0, str(ROOT))
from _nb_common import (
    load_json, gex_day_table, safe_corr, partial_corr_vs_rv, lead_lag_day, SIGNAL_BOARD,
)
from certified_panel import load_certified, primary_days, spot_l2_days, gex_panel_days, DDOI_LABEL, GEX_PROXY_LABEL
from research.lib import squeeze  # noqa: E402

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

def show_pngs(fig_dir: Path, pattern="*.png"):
    if not fig_dir.is_dir():
        print("no figs dir", fig_dir)
        return
    for p in sorted(fig_dir.glob(pattern)):
        display(Markdown(f"**{p.name}** — `{p.relative_to(BOOK)}`"))
        display(Image(filename=str(p)))

def print_board(rows=None):
    rows = rows or SIGNAL_BOARD
    print(f"{'id':34s} {'gate':6s} note")
    print("-" * 88)
    for i, g, n in rows:
        print(f"{i:34s} {g:6s} {n}")

CERT = load_certified()
PRIMARY = primary_days(CERT)
SPOT = spot_l2_days(CERT)
GEX_DAYS = gex_panel_days(CERT)
print("BOOK", BOOK)
print("certified primary n=", len(PRIMARY), PRIMARY)
print("panel_gex_options n=", len(GEX_DAYS), GEX_DAYS)
print("spot_l2 subpanel n=", len(SPOT), SPOT)
print("DDOI label", DDOI_LABEL, "| GEX proxy", GEX_PROXY_LABEL)
print("trade_synth QUARANTINED — never primary; warehouse OI futures-only unused")


In [ ]:

SHADOW = BOOK / "applications" / "paper_shadow" / "out"
meta = load_json(SHADOW / "shadow_meta.json")
board = SHADOW / "SHADOW_BOARD.md"
events = SHADOW / "events.jsonl"
print("shadow_meta", json.dumps(meta, indent=2)[:1200] if meta else None)
if board.exists():
    display(Markdown(board.read_text()[:4000]))
if events.exists():
    print("events.jsonl lines", sum(1 for _ in events.open()))
print("live_orders must remain false; 0 Promote")


## Signal board


In [ ]:

print_board([
    ("risk.gex_exposure", "Hold", "shadow telemetry"),
    ("risk.vex_exposure", "Hold", "shadow telemetry"),
    ("risk.squeeze_intensity", "Hold", "shadow telemetry"),
    ("alpha.tob_cross_arb", "Kill", "never Promote"),
])
